# Tasty Meal — Water (MLS-MPM) on Colab GPU

Runs `tests/test_tasty_meal_water.py` on a Colab GPU with scaled-up geometry and a full cup of water.

## Before you start
1. **Upload your `MLS_MPM_clone` folder to Google Drive** (drag&drop into drive.google.com).
2. **Runtime → Change runtime type → T4 GPU** (or A100 / V100 if available).
3. Run cells top-to-bottom.

## Important
- `.ipynb` itself needs **no extra support** beyond Colab defaults (PyTorch+CUDA preinstalled).
- **Open3D viewer does NOT work on Colab** (no display). Download the result and view locally.
- Colab free session caps at ~12h. `GRID_SIZE=256` may exceed this — start with 128.

## 0. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  dev={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')

## 0.5 Install extra deps
`mpm_pytorch` imports NVIDIA Warp (for SVD) and OmegaConf. Neither is preinstalled on Colab.

In [ ]:
!pip install -q warp-lang omegaconf
import warp as wp, omegaconf
print(f'warp={wp.__version__}   omegaconf={omegaconf.__version__}')

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Point at your code
Adjust `CODE_DIR` if you put the folder somewhere else in Drive.

In [ ]:
CODE_DIR = '/content/drive/MyDrive/MLS_MPM_clone'   # <-- edit if needed

import os, pathlib
os.chdir(CODE_DIR)
print('cwd:', os.getcwd())
assert pathlib.Path('tests/test_tasty_meal_water.py').is_file(), 'script not found, check CODE_DIR'
!ls tests/ | head

## 3. Config — knobs for this run
Start with the **FAST** preset to validate, then switch to **HIGH** for the final render.

In [ ]:
# ============================================================
# CHANGE ME — which preset to run
# ============================================================
PRESET = 'FAST'    # 'FAST' (~10-40 min on T4)  or  'HIGH' (~3-15 h on T4, risky on free Colab)
# ============================================================

PRESETS = {
    # Scaled up 2x, water full, but coarser grid for quick iteration.
    'FAST': dict(
        GRID_SIZE       = 128,
        # Cup: 2x radius and height
        CUP_R_INNER     = 0.120,
        CUP_R_OUTER     = 0.140,
        CUP_Y_BOTTOM    = 0.100,
        CUP_Y_TOP       = 0.340,
        CUP_FLOOR_THK   = 0.024,
        # Water: fill to just under the rim
        WATER_FILL_Y    = 0.330,
        # Ice: 2x size, drop from above the cup
        ICE_HALF        = 0.040,
        ICE_CX          = 0.5,
        ICE_CY          = 0.55,
        ICE_CZ          = 0.5,
        ICE_V0          = (0.0, -7.0, 0.0),
        # Soften water a bit so ice penetrates instead of bouncing
        WATER_K         = 1.0e5,
        ICE_RHO         = 900.0,
    ),
    # Final-quality run. Same scene, but dx=1/256 for sharp contact.
    'HIGH': dict(
        GRID_SIZE       = 256,
        CUP_R_INNER     = 0.120,
        CUP_R_OUTER     = 0.140,
        CUP_Y_BOTTOM    = 0.100,
        CUP_Y_TOP       = 0.340,
        CUP_FLOOR_THK   = 0.024,
        WATER_FILL_Y    = 0.330,
        ICE_HALF        = 0.040,
        ICE_CX          = 0.5,
        ICE_CY          = 0.55,
        ICE_CZ          = 0.5,
        ICE_V0          = (0.0, -7.0, 0.0),
        WATER_K         = 1.0e5,
        ICE_RHO         = 900.0,
    ),
}
CFG = PRESETS[PRESET]
print(f'preset = {PRESET}')
for k, v in CFG.items():
    print(f'  {k} = {v}')

## 4. Patch the Python file
We rewrite only the listed top-of-file constants — every other line is untouched.
The patched file is saved as `tests/_test_tasty_meal_water_patched.py` so your original stays clean.

In [ ]:
import re, shutil, pathlib

SRC = pathlib.Path('tests/test_tasty_meal_water.py')
DST = pathlib.Path('tests/_test_tasty_meal_water_patched.py')
text = SRC.read_text()

def _fmt(v):
    if isinstance(v, tuple):
        return '(' + ', '.join(_fmt(x) for x in v) + ')'
    if isinstance(v, float):
        return repr(v)
    return repr(v)

# scalar single-name constants
for name in ['GRID_SIZE', 'CUP_R_INNER', 'CUP_R_OUTER', 'CUP_Y_BOTTOM',
             'CUP_Y_TOP', 'CUP_FLOOR_THK', 'WATER_FILL_Y',
             'ICE_HALF', 'ICE_V0']:
    if name not in CFG:
        continue
    pat = re.compile(rf'^({re.escape(name)}\s*=\s*)([^\n#]+)(\s*#.*)?$', re.M)
    rep = rf'\g<1>{_fmt(CFG[name])}\g<3>'
    new = pat.sub(rep, text, count=1)
    assert new != text, f'failed to patch {name}'
    text = new

# multi-name tuple lines: ICE_CX, ICE_CY, ICE_CZ
if all(k in CFG for k in ('ICE_CX', 'ICE_CY', 'ICE_CZ')):
    pat = re.compile(r'^(ICE_CX,\s*ICE_CY,\s*ICE_CZ\s*=\s*)([^\n#]+)(\s*#.*)?$', re.M)
    rep = rf"\g<1>{CFG['ICE_CX']}, {CFG['ICE_CY']}, {CFG['ICE_CZ']}\g<3>"
    new = pat.sub(rep, text, count=1)
    assert new != text, 'failed to patch ICE_CX/CY/CZ'
    text = new

# ice material line: E, NU, RHO
if 'ICE_RHO' in CFG:
    pat = re.compile(r'^(ICE_E,\s*ICE_NU,\s*ICE_RHO\s*=\s*)([^\n#]+)(\s*#.*)?$', re.M)
    def _re(m):
        head = m.group(1); tail = m.group(3) or ''
        parts = [p.strip() for p in m.group(2).split(',')]
        parts[2] = repr(CFG['ICE_RHO'])
        return head + ', '.join(parts) + tail
    new = pat.sub(_re, text, count=1)
    assert new != text, 'failed to patch ICE_RHO'
    text = new

# water line: K, GAMMA, RHO
if 'WATER_K' in CFG:
    pat = re.compile(r'^(WATER_K,\s*WATER_GAMMA,\s*WATER_RHO\s*=\s*)([^\n#]+)(\s*#.*)?$', re.M)
    def _rw(m):
        head = m.group(1); tail = m.group(3) or ''
        parts = [p.strip() for p in m.group(2).split(',')]
        parts[0] = repr(CFG['WATER_K'])
        return head + ', '.join(parts) + tail
    new = pat.sub(_rw, text, count=1)
    assert new != text, 'failed to patch WATER_K'
    text = new

# Also retarget the output directory so this run doesn't clobber any prior one
text = re.sub(r'tasty_meal_water_mlsmpm',
              f'tasty_meal_water_mlsmpm_{PRESET.lower()}',
              text)

DST.write_text(text)
print(f'wrote patched file: {DST}')
print('---- patched params (sanity check) ----')
!grep -n -E 'GRID_SIZE|CUP_R_|CUP_Y_|CUP_FLOOR|WATER_FILL|ICE_HALF|ICE_CX|ICE_V0|ICE_E|WATER_K' {DST}

## 5. Run the simulation
Progress prints per frame. Output goes to `tests/output/tasty_meal_water_mlsmpm_<preset>/`.

In [ ]:
# Stream the output live
!python tests/_test_tasty_meal_water_patched.py 2>&1 | tee /content/run.log

## 6. Pack and download results
Tars the output and triggers a browser download. On big runs (`HIGH`) the tarball can be 100s of MB — be patient.

In [ ]:
import pathlib, subprocess
out_name = f'tasty_meal_water_mlsmpm_{PRESET.lower()}'
out_dir  = pathlib.Path('tests/output') / out_name
assert out_dir.is_dir(), f'output missing: {out_dir}'
n_npz = len(list(out_dir.glob('*.npz')))
print(f'packing {n_npz} .npz files from {out_dir}')

tar_path = f'/content/{out_name}.tar.gz'
subprocess.run(['tar', '-czf', tar_path, '-C', 'tests/output', out_name], check=True)
print(f'tar ready: {tar_path}  ({pathlib.Path(tar_path).stat().st_size/1e6:.1f} MB)')

from google.colab import files
files.download(tar_path)

## 7. (Optional) Also save to Drive in case the download fails

In [ ]:
import shutil, pathlib
dest = pathlib.Path('/content/drive/MyDrive') / f'{out_name}.tar.gz'
shutil.copy(tar_path, dest)
print(f'also copied to Drive: {dest}')

## 8. View locally
On your Mac:
```bash
cd ~/Downloads   # or wherever the tarball ended up
tar -xzf tasty_meal_water_mlsmpm_fast.tar.gz

cd /Users/lelinw/Desktop/15763-mpm/MLS_MPM_clone
python tests/view_open3d.py ~/Downloads/tasty_meal_water_mlsmpm_fast --point-size 4
```
Keyboard: SPACE play/pause, ←/→ step, ↑/↓ ±10, R reset, Q quit.